# 23 ExcelWriter 与原生数据透视表示例

本教程演示 `hscredit.excel.ExcelWriter` 的常用 Excel 写入能力，并重点展示两种原生数据透视表数据源：

1. 直接引用当前 `ExcelWriter` 中的 `Worksheet` 和单元格区域；
2. 加载已有 Excel 文件，引用其中的 Sheet 和单元格区域，再另存为新文件。

示例数据完全内置，不依赖网络或外部业务文件。

## Goal

完成本教程后，将得到以下文件：

- `23_excel_pivot_from_worksheet.xlsx`：同一个 Writer 中写入明细并创建透视表；
- `23_excel_pivot_source.xlsx`：仅包含明细数据的已有 Excel；
- `23_excel_pivot_from_existing.xlsx`：加载已有 Excel 后创建透视表并另存。

数据透视表是 Excel 可交互、可刷新的原生对象，不是普通聚合结果表。

## Setup

导入依赖并确定仓库与输出路径。Notebook 可以从仓库根目录或 `examples/` 目录执行。

In [1]:
from pathlib import Path
import sys
import zipfile
import xml.etree.ElementTree as ET

import pandas as pd
from IPython.display import display
from openpyxl import load_workbook
from openpyxl.utils import range_boundaries

working_directory = Path.cwd().resolve()
repo_root = next(
    (
        candidate
        for candidate in [working_directory, *working_directory.parents]
        if (candidate / "hscredit").is_dir() and (candidate / "examples").is_dir()
    ),
    None,
)
if repo_root is None:
    raise RuntimeError("未找到 hscredit 仓库根目录，请从仓库或 examples 目录执行本 Notebook")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from hscredit.excel import ExcelWriter, dataframe2excel

examples_dir = repo_root / "examples"
worksheet_output_path = examples_dir / "23_excel_pivot_from_worksheet.xlsx"
existing_source_path = examples_dir / "23_excel_pivot_source.xlsx"
existing_output_path = examples_dir / "23_excel_pivot_from_existing.xlsx"

# 保证重复执行结果一致，仅清理本 Notebook 自己生成的三个文件。
for generated_path in (worksheet_output_path, existing_source_path, existing_output_path):
    generated_path.unlink(missing_ok=True)

print("输出目录：", examples_dir.name)

输出目录： examples


## Data

使用一份小型贷款明细数据，字段同时覆盖文本、月份、金额和计数，便于演示筛选、条件格式和多值透视。

In [2]:
loan_data = pd.DataFrame(
    {
        "商品类别": ["数码", "服饰", "食品", "数码", "服饰", "食品", "数码", "服饰", "食品", "数码", "服饰", "食品"],
        "区域": ["华东", "华东", "华东", "华南", "华南", "华南", "华北", "华北", "华北", "华东", "华南", "华北"],
        "客户类型": ["新客", "老客", "新客", "老客", "新客", "老客", "新客", "老客", "新客", "老客", "新客", "老客"],
        "月份": ["2026-01"] * 6 + ["2026-02"] * 6,
        "放款金额": [120000, 85000, 46000, 150000, 92000, 51000, 132000, 88000, 49000, 160000, 98000, 55000],
        "笔数": [12, 9, 7, 15, 10, 8, 13, 9, 7, 16, 11, 8],
    }
)

assert loan_data.shape == (12, 6)
display(loan_data)

,商品类别,区域,客户类型,月份,放款金额,笔数
0,数码,华东,新客,2026-01,120000,12
1,服饰,华东,老客,2026-01,85000,9
2,食品,华东,新客,2026-01,46000,7
3,数码,华南,老客,2026-01,150000,15
4,服饰,华南,新客,2026-01,92000,10
5,食品,华南,老客,2026-01,51000,8
6,数码,华北,新客,2026-02,132000,13
7,服饰,华北,老客,2026-02,88000,9
8,食品,华北,新客,2026-02,49000,7
9,数码,华东,老客,2026-02,160000,16


## Steps

### 1. 从当前 Worksheet 创建透视表

先在同一个 `ExcelWriter` 中创建“贷款明细”工作表并写入数据，再直接把该 `Worksheet` 的 `A3:F15` 作为透视表数据源。`filter_items` 作用于已有的“区域”轴：华东、华南可见，华北仅在透视表中隐藏，源数据不会被删除。

In [3]:
worksheet_writer = ExcelWriter(theme_color="2639E9", condition_color="F76E6C")
detail_sheet = worksheet_writer.get_sheet_by_name("贷款明细")

worksheet_writer.insert_value2sheet(
    detail_sheet,
    "A1",
    value="贷款明细（Worksheet 数据源）",
    style="header",
    end_space="F1",
)
worksheet_writer.insert_df2sheet(
    detail_sheet,
    loan_data,
    "A3",
    header=True,
    index=False,
    auto_width=True,
    fill=True,
)
worksheet_writer.set_freeze_panes(detail_sheet, "A4")
worksheet_writer.add_auto_filter(detail_sheet, "A3:F15")
worksheet_writer.add_conditional_formatting(detail_sheet, "E4", "E15")
worksheet_writer.set_number_format(detail_sheet, "E4:E15", "#,##0")
worksheet_writer.insert_hyperlink2sheet(
    detail_sheet,
    "H2",
    sheet="透视分析",
    target_space="B2",
)

pivot_sheet = worksheet_writer.get_sheet_by_name("透视分析")
worksheet_writer.insert_hyperlink2sheet(
    pivot_sheet,
    "A1",
    sheet="贷款明细",
    target_space="A1",
)
worksheet_writer.insert_pivot_table2sheet(
    worksheet=pivot_sheet,
    pivot_anchor="B2",
    source_sheet=detail_sheet,
    source_range="A3:F15",
    rows="商品类别",
    columns="区域",
    filter_items={"区域": ["华东", "华南"]},
    values=[
        {"field": "放款金额", "agg": "sum", "name": "放款金额", "number_format": "#,##0"},
        {
            "field": "放款金额",
            "agg": "sum",
            "show_as": "全局占比",
            "name": "金额占比",
            "number_format": "0.00%",
        },
    ],
    name="Worksheet数据透视表",
)
worksheet_writer.save(str(worksheet_output_path))

print("已生成：", worksheet_output_path.name)

已生成： 23_excel_pivot_from_worksheet.xlsx


### 2. 从已有 Excel 文件创建透视表

先用 `dataframe2excel` 生成一个只有“贷款明细”的现有文件，再通过 `style_excel` 加载它。由于文件中没有“初始化”模板 Sheet，新增的“透视分析”会直接创建为空白 Sheet，原业务 Sheet 保持不变。

In [4]:
dataframe2excel(
    loan_data,
    str(existing_source_path),
    sheet_name="贷款明细",
    index=False,
    start_row=1,
    start_col=1,
    fill=True,
    auto_filter=True,
    condition_cols=["放款金额"],
    color_cols=["笔数"],
    custom_cols=["放款金额", "笔数"],
    custom_format="#,##0",
    theme_color="2639E9",
    condition_color="F76E6C",
)

source_workbook = load_workbook(existing_source_path)
assert source_workbook.sheetnames == ["贷款明细"]
source_workbook.close()

print("已有数据文件：", existing_source_path.name)

已有数据文件： 23_excel_pivot_source.xlsx


In [5]:
existing_writer = ExcelWriter(
    style_excel=str(existing_source_path),
    mode="append",
    theme_color="3F1DBA",
)
existing_pivot_sheet = existing_writer.get_sheet_by_name("透视分析")

existing_writer.insert_hyperlink2sheet(
    existing_pivot_sheet,
    "A1",
    sheet="贷款明细",
    target_space="A1",
)
existing_writer.insert_pivot_table2sheet(
    worksheet=existing_pivot_sheet,
    pivot_anchor="B2",
    source_sheet="贷款明细",
    source_range="A1:F13",
    rows=["商品类别", "客户类型"],
    columns="区域",
    filter_items={"区域": ["华东", "华南"]},
    values=[
        {"field": "放款金额", "agg": "sum", "name": "放款金额", "number_format": "#,##0"},
        {"field": "笔数", "agg": "sum", "name": "贷款笔数", "number_format": "#,##0"},
    ],
    subtotals=True,
    name="已有文件数据透视表",
)
existing_writer.save(str(existing_output_path))

# 如需原地保存，可将上一行改为：existing_writer.save(str(existing_source_path))
print("已另存：", existing_output_path.name)

已另存： 23_excel_pivot_from_existing.xlsx


## Checks

使用 openpyxl 与 OOXML 双重检查：

- 明细数据没有被透视表流程改写；
- 输出工作表含有原生 PivotTable 对象；
- xlsx 内包含 pivotTable、pivotCacheDefinition 和 pivotCacheRecords 部件；
- 数据源 Sheet 与区域引用准确。

In [6]:
OOXML_NS = {"main": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}


def dataframe_from_range(worksheet, cell_range):
    """把含表头的工作表区域读取为 DataFrame。"""
    min_col, min_row, max_col, max_row = range_boundaries(cell_range)
    values = list(
        worksheet.iter_rows(
            min_row=min_row,
            max_row=max_row,
            min_col=min_col,
            max_col=max_col,
            values_only=True,
        )
    )
    return pd.DataFrame(values[1:], columns=values[0])


def inspect_native_pivot(
    workbook_path,
    pivot_sheet_name,
    expected_source_sheet,
    expected_source_ref,
    expected_data,
):
    """检查完整源数据、原生透视对象、筛选项和 OOXML 数据源引用。"""
    workbook = load_workbook(workbook_path)
    try:
        pivot_count = len(workbook[pivot_sheet_name]._pivots)
        sheet_names = workbook.sheetnames
        actual_data = dataframe_from_range(
            workbook[expected_source_sheet],
            expected_source_ref,
        )
    finally:
        workbook.close()

    pd.testing.assert_frame_equal(
        actual_data.reset_index(drop=True),
        expected_data.reset_index(drop=True),
        check_dtype=False,
    )

    with zipfile.ZipFile(workbook_path) as archive:
        part_names = archive.namelist()
        pivot_parts = sorted(
            name
            for name in part_names
            if name.startswith("xl/pivotTables/pivotTable") and name.endswith(".xml")
        )
        cache_definition_parts = sorted(
            name
            for name in part_names
            if name.startswith("xl/pivotCache/pivotCacheDefinition") and name.endswith(".xml")
        )
        cache_record_parts = sorted(
            name
            for name in part_names
            if name.startswith("xl/pivotCache/pivotCacheRecords") and name.endswith(".xml")
        )
        assert pivot_parts and cache_definition_parts and cache_record_parts
        cache_root = ET.fromstring(archive.read(cache_definition_parts[0]))
        pivot_root = ET.fromstring(archive.read(pivot_parts[0]))

    worksheet_source = cache_root.find("main:cacheSource/main:worksheetSource", OOXML_NS)
    assert worksheet_source is not None
    actual_source_ref = worksheet_source.attrib["ref"]
    actual_source_sheet = worksheet_source.attrib["sheet"]

    cache_fields = cache_root.findall("main:cacheFields/main:cacheField", OOXML_NS)
    cache_field_names = [field.attrib["name"] for field in cache_fields]
    region_field_index = cache_field_names.index("区域")
    shared_items = cache_fields[region_field_index].find("main:sharedItems", OOXML_NS)
    region_values = [item.attrib.get("v") for item in shared_items]

    pivot_fields = pivot_root.find("main:pivotFields", OOXML_NS)
    region_items = pivot_fields[region_field_index].find("main:items", OOXML_NS)
    visible_regions = []
    hidden_regions = []
    for item in region_items:
        shared_index = item.attrib.get("x")
        if shared_index is None or int(shared_index) >= len(region_values):
            continue
        target = hidden_regions if item.attrib.get("h") == "1" else visible_regions
        target.append(region_values[int(shared_index)])

    assert pivot_count == 1
    assert cache_root.attrib.get("refreshOnLoad") == "1"
    assert actual_source_sheet == expected_source_sheet
    assert actual_source_ref == expected_source_ref
    assert set(visible_regions) == {"华东", "华南"}
    assert set(hidden_regions) == {"华北"}

    return {
        "文件": workbook_path.name,
        "工作表": "、".join(sheet_names),
        "原生透视表数": pivot_count,
        "源数据": f"{actual_source_sheet}!{actual_source_ref}",
        "源数据完整一致": True,
        "筛选后区域": "、".join(visible_regions),
    }

In [7]:
worksheet_check = inspect_native_pivot(
    worksheet_output_path,
    pivot_sheet_name="透视分析",
    expected_source_sheet="贷款明细",
    expected_source_ref="A3:F15",
    expected_data=loan_data,
)
existing_check = inspect_native_pivot(
    existing_output_path,
    pivot_sheet_name="透视分析",
    expected_source_sheet="贷款明细",
    expected_source_ref="A1:F13",
    expected_data=loan_data,
)

original_workbook = load_workbook(existing_source_path)
output_workbook = load_workbook(existing_output_path)
try:
    assert original_workbook.sheetnames == ["贷款明细"]
    original_data = dataframe_from_range(original_workbook["贷款明细"], "A1:F13")
    output_data = dataframe_from_range(output_workbook["贷款明细"], "A1:F13")
    pd.testing.assert_frame_equal(original_data, output_data, check_dtype=False)
finally:
    original_workbook.close()
    output_workbook.close()

verification = pd.DataFrame([worksheet_check, existing_check])
display(verification)

,文件,工作表,原生透视表数,源数据,源数据完整一致,筛选后区域
0,23_excel_pivot_from_worksheet.xlsx,贷款明细、透视分析,1,贷款明细!A3:F15,True,华东、华南
1,23_excel_pivot_from_existing.xlsx,贷款明细、透视分析,1,贷款明细!A1:F13,True,华东、华南


## Next Steps

- 在 Microsoft Excel 中打开生成文件时，透视缓存会自动刷新，可继续拖拽字段、切换筛选项或修改样式。
- 若要原地更新已有文件，把 `save` 路径改为 `style_excel` 的同一路径即可。
- `rows`、`columns`、`values`、`filter_items`、`groups`、`subtotals` 和汇总/占比配置可以按业务需要组合。
- `source_range` 必须包含首行字段名，且字段名不能为空或重复。